# ChronoPDE V2 Phase 5 — five-seed development study

Attach the private Phase 3 dataset containing `chronopde_v2_development.h5`, enable Internet, and select **GPU T4 x2**. The notebook trains fresh matched FFT and DCT models for seeds 0–4. Each seed runs both models concurrently, one per GPU. It never generates or reads confirmatory data and it does not authorize a superiority claim.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import torch

REPOSITORY_URL = 'https://github.com/madhavkapoor13/ChronoPDE.git'
BRANCH = 'codex/chronopde-v2-phase5'
REPOSITORY = Path('/kaggle/working/ChronoPDE')
if not REPOSITORY.exists():
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--single-branch', REPOSITORY_URL, str(REPOSITORY)],
        check=True,
    )
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '-e', str(REPOSITORY)],
    check=True,
)
COMMIT = subprocess.check_output(
    ['git', '-C', str(REPOSITORY), 'rev-parse', 'HEAD'], text=True
).strip()
GPU_COUNT = torch.cuda.device_count()
assert GPU_COUNT >= 1, 'Enable a Kaggle GPU accelerator before running Phase 5.'
print('Commit:', COMMIT)
print('PyTorch:', torch.__version__)
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)])

In [ ]:
candidates = list(Path('/kaggle/input').rglob('chronopde_v2_development.h5'))
assert len(candidates) == 1, f'Expected exactly one Phase 3 HDF5, found: {candidates}'
DATA = candidates[0]
print('Dataset:', DATA, f'({DATA.stat().st_size / 2**30:.2f} GiB)')
check_command = [
    sys.executable, 'scripts/chronopde_v2.py', 'phase5',
    '--data-path', str(DATA), '--check-only', '--format', 'json',
]
subprocess.run(check_command, cwd=REPOSITORY, check=True)

In [ ]:
# Optional recovery: attach a prior Kaggle output dataset containing this ZIP.
resume_archives = list(Path('/kaggle/input').rglob('chronopde_v2_phase5_partial.zip'))
assert len(resume_archives) <= 1, f'Attach at most one recovery archive: {resume_archives}'
if resume_archives:
    with zipfile.ZipFile(resume_archives[0]) as archive:
        root = REPOSITORY.resolve()
        for info in archive.infolist():
            destination = (root / info.filename).resolve()
            assert destination == root or root in destination.parents, info.filename
        archive.extractall(REPOSITORY)
    print('Restored:', resume_archives[0])
else:
    print('Starting fresh; no recovery archive attached.')

In [ ]:
def run_one(model, seed, gpu):
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(gpu)
    command = [
        sys.executable, 'scripts/chronopde_v2.py', 'phase5',
        '--data-path', str(DATA), '--model', model, '--seed', str(seed),
        '--device', 'cuda', '--format', 'json',
    ]
    prefix = f'chronopde_v2-p5-reaction_diffusion-exact_rhs-{model}-multiseed-s{seed}-'
    matches = list((REPOSITORY / 'artifacts/chronopde_v2/runs').glob(prefix + '*'))
    if matches:
        manifest_path = matches[0] / 'run_manifest.json'
        if manifest_path.is_file():
            status = json.loads(manifest_path.read_text())['execution_status']
            if status == 'running':
                command.append('--resume')
            elif status in ('failed', 'interrupted'):
                message = f'{model} seed {seed} is finalized as {status}'
                raise RuntimeError(message + '; use a fresh notebook output.')
    process = subprocess.Popen(
        command, cwd=REPOSITORY, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(f'[{model} s{seed} gpu{gpu}] {line}', end='', flush=True)
    return model, seed, process.wait()

def safe_run(job):
    model, seed, _ = job
    try:
        return run_one(*job)
    except Exception as error:
        print(f'[{model} s{seed}] FAILED: {type(error).__name__}: {error}', flush=True)
        return model, seed, 99

return_codes = {}
for seed in range(5):
    jobs = [('fft', seed, 0), ('dct', seed, 1 if GPU_COUNT >= 2 else 0)]
    if GPU_COUNT >= 2:
        with ThreadPoolExecutor(max_workers=2) as executor:
            futures = [executor.submit(safe_run, job) for job in jobs]
            for future in as_completed(futures):
                model, completed_seed, code = future.result()
                return_codes[(model, completed_seed)] = code
    else:
        print('Only one GPU detected; running the matched pair sequentially.')
        for job in jobs:
            model, completed_seed, code = safe_run(job)
            return_codes[(model, completed_seed)] = code
print('Training return codes:', return_codes)

In [ ]:
successful = len(return_codes) == 10 and all(code == 0 for code in return_codes.values())
if successful:
    collect_command = [
        sys.executable, 'scripts/chronopde_v2.py', 'phase5', '--collect', '--format', 'json',
    ]
    collection = subprocess.run(
        collect_command, cwd=REPOSITORY, text=True, capture_output=True
    )
    print(collection.stdout)
    successful = collection.returncode == 0
if successful:
    package_root = REPOSITORY / 'artifacts/chronopde_v2/runs'
    packages = list(package_root.rglob('chronopde_v2_phase5_outputs.zip'))
    successful = len(packages) == 1
if successful:
    FINAL_OUTPUT = Path('/kaggle/working/chronopde_v2_phase5_outputs.zip')
    shutil.copy2(packages[0], FINAL_OUTPUT)
    assert zipfile.ZipFile(FINAL_OUTPUT).testzip() is None
    print('Download:', FINAL_OUTPUT, f'({FINAL_OUTPUT.stat().st_size / 2**20:.1f} MiB)')
    print('The ZIP contains reports, plots, metrics, manifests,')
    print('and all 10 selected best checkpoints.')
else:
    PARTIAL = Path('/kaggle/working/chronopde_v2_phase5_partial.zip')
    roots = [
        REPOSITORY / 'artifacts/chronopde_v2/runs',
        REPOSITORY / 'reports/chronopde_v2/phase5',
    ]
    with zipfile.ZipFile(PARTIAL, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for source_root in roots:
            if source_root.exists():
                for path in source_root.rglob('*'):
                    if path.is_file() and '.tmp' not in path.name:
                        archive.write(path, path.relative_to(REPOSITORY))
    assert zipfile.ZipFile(PARTIAL).testzip() is None
    print('Download recovery package:', PARTIAL)
assert successful, f'One or more runs failed; preserve the partial ZIP: {return_codes}'